# 데이터전처리_함수설명

각 설명 다음의 코드 셀을 Shift+Enter로 실행하세요. 이 노트북은 함수를 정의하고 내부 모듈을 준비합니다. 실제 전처리·학습 호출은 **00번 통합 노트북**에서 합니다. 내부 파일은 셀에서 자동 생성하므로 별도 Python 파일을 준비할 필요가 없습니다.

In [ ]:
from pathlib import Path
import sys, json
# 통합 노트북은 이 폴더를 작업 폴더로 지정합니다. 개별 실행도 같은 폴더에서 하세요.
WORKSPACE = Path.cwd().resolve()
RUNTIME = WORKSPACE / '_runtime'
(RUNTIME / 'gnn_pipeline').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'gnn_pipeline' / '__init__.py').touch()
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


## 실행 셀을 내부 모듈로 연결

아래 함수는 이 노트북의 코드 셀을 그대로 저장해 다음 단계에서 import할 수 있게 합니다.

In [ ]:
def export_cells(notebook_name, module_name):
    """이 노트북의 실제 코드 셀을 모아 내부 import용 모듈을 생성합니다."""
    notebook = json.loads((WORKSPACE / notebook_name).read_text(encoding='utf-8'))
    blocks = [cell['source'] for cell in notebook['cells']
              if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('module') == module_name]
    source = '\n\n'.join(''.join(block) if isinstance(block, list) else block for block in blocks) + '\n'
    path = RUNTIME / 'gnn_pipeline' / (module_name + '.py')
    compile(source, str(path), 'exec')
    path.write_text(source, encoding='utf-8')
    print('내부 모듈 준비:', path.name)
    return path


## 라이브러리·상수 준비

다음 함수가 사용하는 라이브러리와 설정을 준비합니다.

In [ ]:
"""Daily-close AML pipeline. Offline labels are never input features.

Input dates are half-open. A scoring day sees that day and prior days only.
Full builds are explicit: importing this module performs no data processing.
"""
from __future__ import annotations
import csv, hashlib, json, re, sys
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
ROOT = Path.cwd()
RAW_COLUMNS = ['timestamp_raw', 'from_bank', 'from_account', 'to_bank', 'to_account', 'amount_received_raw', 'receiving_currency', 'amount_paid_raw', 'payment_currency', 'payment_format', 'label_raw']
HEADERS = ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
ACCOUNT_HEADERS = ['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']
ACCOUNT_COLUMNS = ['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name']
ENTITY_TYPES = ['Corporation', 'Partnership', 'Sole Proprietorship', 'Individual', 'Country', 'Direct']
EDGE_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'sender_gap_minutes', 'sender_count_1h', 'receiver_gap_minutes', 'receiver_count_1h', 'paid_currency_z', 'pair_history_count', 'reverse_history_count', 'pair_age_at_day_start', 'same_entity', 'same_known_country', 'both_countries_known', 'same_currency', 'is_self_transfer', 'paid_currency_unseen', 'received_currency_unseen']
NODE_FEATURES = ['log_out_count', 'log_in_count', 'log_out_neighbors', 'log_in_neighbors', 'log_self_count', 'log_reciprocal_neighbors', 'net_count_ratio', 'reciprocity_ratio', 'mean_paid_currency_z', 'mean_received_currency_z', 'paid_outlier_fraction', 'received_outlier_fraction', 'log_currency_count', 'log_active_days'] + ['type_' + x for x in ENTITY_TYPES] + ['log_entity_observed_accounts', 'log_entity_out_diversity', 'log_entity_in_diversity']
assert len(NODE_FEATURES) == len(set(NODE_FEATURES)) == 23
assert len(EDGE_FEATURES) == len(set(EDGE_FEATURES)) == 18


## `Config`

원본 데이터 위치, 날짜 경계, 이웃 수, 학습 설정을 한 곳에 보관합니다. validate는 날짜와 설정값을 검사합니다.

**메서드:** `validate`. 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
@dataclass(frozen=True)
class Config:
    source: str = 'data'
    output: str = str(ROOT / 'outputs' / 'data')
    calibration_start: str = '2022-08-01'
    train_start: str = '2022-08-08'
    val_start: str = '2022-09-01'
    test_start: str = '2022-10-01'
    test_end: str = '2022-11-01'
    history_days: int = 30
    incoming_neighbors: int = 10
    target_batch_size: int = 4096
    duplicate_policy: str = 'keep'
    memory_limit: str = '3GB'
    threads: int = 2
    epochs: int = 5
    patience: int = 2
    hidden: int = 128
    heads: int = 2
    lr: float = 0.002
    pos_weight: float = 300.0
    seed: int = 42

    def validate(self):
        dates = [pd.Timestamp(getattr(self, x)) for x in ['calibration_start', 'train_start', 'val_start', 'test_start', 'test_end']]
        assert all((a < b for a, b in zip(dates, dates[1:]))), 'Date boundaries must be strictly increasing'
        assert self.duplicate_policy in {'keep', 'drop_exact'}
        assert self.history_days >= 1 and self.incoming_neighbors >= 1 and (self.target_batch_size >= 1)
        assert self.epochs >= 1 and self.patience >= 1 and (self.hidden % self.heads == 0)

## `sql_string`

문자열을 SQL 문자열 리터럴로 안전하게 이스케이프합니다.

**입력:** `x`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `"'" + str(x).replace("'", "''") + "'"`

In [ ]:
def sql_string(x):
    return "'" + str(x).replace("'", "''") + "'"

## `file_fingerprint`

파일 크기·수정 시각·앞뒤 일부 바이트 해시로 원본 변경 여부를 확인할 정보를 만듭니다.

**입력:** `p`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `{'path': str(p.resolve()), 'bytes': s.st_size, 'mtime_ns': s.st_mtime_ns, 'head_tail_sha256': hashlib.sha256(head + tail).hexdigest()}`

In [ ]:
def file_fingerprint(p):
    p = Path(p)
    s = p.stat()
    with p.open('rb') as f:
        head = f.read(1024 * 1024)
        f.seek(max(0, s.st_size - 1024 * 1024))
        tail = f.read()
    return {'path': str(p.resolve()), 'bytes': s.st_size, 'mtime_ns': s.st_mtime_ns, 'head_tail_sha256': hashlib.sha256(head + tail).hexdigest()}

## `json_write`

JSON을 임시 파일에 저장한 뒤 교체해 중간 파일을 완료 결과로 오인하지 않게 합니다.

**입력:** `p`, `data`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** 명시적 반환값 없이 파일 저장·상태 변경을 수행합니다. 위 역할 설명을 확인하세요.

In [ ]:
def json_write(p, data):
    p = Path(p)
    tmp = p.with_suffix(p.suffix + '.tmp')
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    tmp.replace(p)

## `base`

출력 폴더를 만들고 설정값을 검사합니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `p`

In [ ]:
def base(cfg):
    cfg.validate()
    p = Path(cfg.output)
    p.mkdir(parents=True, exist_ok=True)
    return p

## `connect`

DuckDB에 연결하고 메모리 한도와 스레드 수를 적용합니다. 읽기 전용 연결도 지원합니다.

**입력:** `cfg`, `read_only`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `c`

In [ ]:
def connect(cfg, read_only=False):
    p = base(cfg)
    c = duckdb.connect(str(p / 'daily.duckdb'), read_only=read_only)
    c.execute(f'SET memory_limit={sql_string(cfg.memory_limit)}')
    c.execute(f'SET threads={cfg.threads}')
    c.execute(f'SET temp_directory={sql_string(p / 'spill')}')
    c.execute('SET preserve_insertion_order=true')
    return c

## `stage_manifest`

단계의 완료 상태와 입력·코드 정보를 JSON으로 기록합니다.

**입력:** `cfg`, `stage`, `details`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** 명시적 반환값 없이 파일 저장·상태 변경을 수행합니다. 위 역할 설명을 확인하세요.

In [ ]:
def stage_manifest(cfg, stage, details):
    json_write(base(cfg) / (stage + '.json'), {'stage': stage, 'complete': True, 'config': asdict(cfg), 'code_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), **details})

## `require_stage`

선행 단계의 완료 여부·설정·코드·원본 파일 지문이 일치하는지 확인합니다.

**입력:** `cfg`, `stage`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `m`

In [ ]:
def require_stage(cfg, stage):
    p = base(cfg) / (stage + '.json')
    assert p.exists(), f'Run {stage} first'
    m = json.loads(p.read_text())
    assert m.get('complete'), f'Incomplete {stage}'
    assert m['config'] == asdict(cfg), 'Configuration changed: use a new output directory or rebuild stages'
    assert m['code_sha256'] == hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), 'Code changed: rebuild into a new output directory'
    if 'sources' in m:
        for spec in m['sources'].values():
            assert file_fingerprint(spec['path']) == spec, 'Source changed; use new output directory'
    return m

## `fresh_stage`

이미 완료된 전처리 단계를 실수로 덮어쓰지 않도록 검사합니다.

**입력:** `cfg`, `name`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** 명시적 반환값 없이 파일 저장·상태 변경을 수행합니다. 위 역할 설명을 확인하세요.

In [ ]:
def fresh_stage(cfg, name):
    p = base(cfg) / (name + '.json')
    if p.exists():
        raise FileExistsError(f'{name} already completed. Use a new output directory; do not overwrite dependencies.')

## `finite`

배열에 NaN이나 무한대가 없는지 검사하고 문제가 있으면 위치를 알려줍니다.

**입력:** `name`, `x`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `x`

In [ ]:
def finite(name, x):
    x = np.asarray(x)
    bad = ~np.isfinite(x)
    if bad.any():
        example = np.argwhere(bad)[:5].tolist()
        raise ValueError(f'{name}: {int(bad.sum())} NaN/Inf values, first positions={example}')
    return x

## `_country_of`

은행 이름에서 지원하는 국가 코드를 추출합니다.

**입력:** `name`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `'UNKNOWN'` / `value or 'UNKNOWN'`

In [ ]:
def _country_of(name):
    m = re.match('^(.*?) Bank #', str(name))
    value = m.group(1).strip() if m else 'UNKNOWN'
    if value.lower() in {'crypto', 'crytpo'}:
        return 'UNKNOWN'
    return value or 'UNKNOWN'

## `build_dataset`

CSV 열과 값 검사 → 완전 중복 처리 → 계좌 ID 매핑 → 거래 정규화 → 감사 결과 저장 순서로 전처리합니다. 원본 파일은 수정하지 않습니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `summary`

In [ ]:
def build_dataset(cfg=Config()):
    """Strict original-column ingest, exact duplicate audit, normalized KYC join.
    Fresh output required. Durable edge_id is original zero-based CSV row position.
    """
    fresh_stage(cfg, 'dataset')
    p = base(cfg)
    src = Path(cfg.source)
    files = {'transactions': src / 'HI-Large_Trans.csv', 'accounts': src / 'HI-Large_accounts.csv', 'patterns': src / 'HI-Large_Patterns.txt'}
    specs = {k: file_fingerprint(v) for k, v in files.items()}
    for key, expected in [('transactions', HEADERS), ('accounts', ACCOUNT_HEADERS)]:
        with files[key].open(newline='') as f:
            actual = next(csv.reader(f))
        if actual != expected:
            raise ValueError(f'{key}: unexpected column order/header: {actual}')
    c = connect(cfg)
    try:
        cols = '{' + ','.join((sql_string(x) + ":'VARCHAR'" for x in RAW_COLUMNS)) + '}'
        ingest_marker = p / 'raw_ingest.json'
        if ingest_marker.exists():
            ing = json.loads(ingest_marker.read_text())
            assert ing['sources'] == specs, 'Source changed since raw ingestion'
            assert ing['raw_columns'] == RAW_COLUMNS and ing['parser_version'] == 1
            assert c.execute('SELECT count(*) FROM raw').fetchone()[0] == ing['rows']
        else:
            c.execute(f"CREATE OR REPLACE TABLE raw AS SELECT row_number() OVER ()-1 AS edge_id,*\n              FROM read_csv({sql_string(files['transactions'])},header=true,columns={cols},\n              parallel=false,strict_mode=true,nullstr='')")
            c.execute('CHECKPOINT')
            json_write(ingest_marker, {'sources': specs, 'rows': c.execute('SELECT count(*) FROM raw').fetchone()[0], 'raw_columns': RAW_COLUMNS, 'parser_version': 1})
        invalid = []
        for x in RAW_COLUMNS:
            invalid.append(f"{x} IS NULL OR trim({x})='' OR lower(trim({x})) IN ('null','none','nan','na','n/a','?') OR {x}!=trim({x})")
        invalid += [f"NOT regexp_full_match({x},'[0-9]+') OR try_cast({x} AS BIGINT) IS NULL" for x in ['from_bank', 'to_bank']]
        invalid += [f"NOT regexp_full_match({x},'[0-9A-Fa-f]+')" for x in ['from_account', 'to_account']]
        invalid += [f"NOT regexp_full_match({x},'[0-9]+([.][0-9]{{1,8}})?') OR coalesce(try_cast({x} AS DECIMAL(18,8))::DECIMAL(38,8),try_cast({x} AS DECIMAL(38,8))) IS NULL" for x in ['amount_paid_raw', 'amount_received_raw']]
        invalid += ["label_raw NOT IN ('0','1')", "try_strptime(timestamp_raw,'%Y/%m/%d %H:%M') IS NULL"]
        cond = ' OR '.join(('(' + x + ')' for x in invalid))
        count = c.execute('SELECT count(*) FROM raw WHERE ' + cond).fetchone()[0]
        if count:
            c.execute('SELECT * FROM raw WHERE ' + cond + ' LIMIT 50').fetchdf().to_csv(p / 'invalid_raw_examples.csv', index=False)
            raise ValueError(f'{count} invalid source rows; see invalid_raw_examples.csv')
        raw_cols = ','.join(RAW_COLUMNS)
        c.execute(f'CREATE OR REPLACE TABLE raw_hash AS SELECT edge_id,hash({raw_cols}) AS h FROM raw')
        c.execute('CREATE OR REPLACE TEMP TABLE candidate_hash AS SELECT h FROM raw_hash GROUP BY h HAVING count(*)>1')
        c.execute(f'CREATE OR REPLACE TABLE duplicate_groups AS SELECT {','.join(('r.' + x for x in RAW_COLUMNS))},\n            min(r.edge_id) AS first_edge_id,count(*) AS multiplicity\n            FROM raw r JOIN raw_hash h USING(edge_id) JOIN candidate_hash q USING(h)\n            GROUP BY {','.join(('r.' + x for x in RAW_COLUMNS))} HAVING count(*)>1')
        join = ' AND '.join(('r.' + x + '=d.' + x for x in RAW_COLUMNS))
        c.execute(f'CREATE OR REPLACE TABLE duplicate_rows AS SELECT r.edge_id,d.first_edge_id AS duplicate_group_id,\n            d.multiplicity FROM raw r JOIN raw_hash h USING(edge_id) JOIN candidate_hash q USING(h)\n            JOIN duplicate_groups d ON {join}')
        c.execute('SELECT * FROM duplicate_rows ORDER BY edge_id').fetchdf().to_csv(p / 'duplicate_audit.csv', index=False)
        c.execute(f'CREATE OR REPLACE TABLE retained AS SELECT r.*,d.duplicate_group_id\n            FROM raw r LEFT JOIN duplicate_rows d USING(edge_id)\n            {('WHERE d.duplicate_group_id IS NULL OR r.edge_id=d.duplicate_group_id' if cfg.duplicate_policy == 'drop_exact' else '')}')
        bank = lambda x: f'cast(cast({x} AS BIGINT) AS VARCHAR)'
        c.execute(f'CREATE OR REPLACE TABLE nodes AS SELECT row_number() OVER(ORDER BY bank,account)-1 AS node_id,bank,account FROM\n          (SELECT {bank('from_bank')} AS bank,from_account AS account FROM retained UNION\n           SELECT {bank('to_bank')},to_account FROM retained)')
        acols = '{' + ','.join((sql_string(x) + ":'VARCHAR'" for x in ACCOUNT_COLUMNS)) + '}'
        c.execute(f"CREATE OR REPLACE TABLE accounts_raw AS SELECT * FROM read_csv({sql_string(files['accounts'])},header=true,\n            columns={acols},parallel=false,strict_mode=true,nullstr='')")
        acond = ' OR '.join((f"{x} IS NULL OR trim({x})='' OR {x}!=trim({x})" for x in ACCOUNT_COLUMNS))
        acond += " OR NOT regexp_full_match(bank_id,'[0-9]+') OR try_cast(bank_id AS BIGINT) IS NULL OR NOT regexp_full_match(account_number,'[0-9A-Fa-f]+')"
        assert c.execute('SELECT count(*) FROM accounts_raw WHERE ' + acond).fetchone()[0] == 0, 'Invalid account registry'
        c.execute(f'CREATE OR REPLACE TABLE accounts AS SELECT *,{bank('bank_id')} AS bank FROM accounts_raw')
        conflicts = c.execute('SELECT count(*) FROM (SELECT bank,account_number FROM accounts GROUP BY 1,2 HAVING count(*)>1)').fetchone()[0]
        assert conflicts == 0, f'{conflicts} duplicate registry keys: resolve explicitly, never choose first silently'
        c.execute('CREATE OR REPLACE TABLE kyc AS SELECT n.*,a.entity_id,a.entity_name,a.bank_name,\n          CASE WHEN a.entity_id IS NULL THEN 1 ELSE 0 END AS kyc_missing,\n          coalesce(ec.acct_count,0) AS acct_count\n          FROM nodes n LEFT JOIN accounts a ON n.bank=a.bank AND n.account=a.account_number\n          LEFT JOIN (SELECT entity_id,count(*) AS acct_count FROM accounts GROUP BY entity_id) ec ON a.entity_id=ec.entity_id')
        k = c.execute('SELECT * FROM kyc ORDER BY node_id').fetchdf()
        k['country'] = k.bank_name.map(_country_of)
        k['entity_type'] = k.entity_name.str.extract('^([A-Za-z ]+?)\\s*#')[0].fillna('UNKNOWN')
        c.register('kyc_final', k)
        c.execute('CREATE OR REPLACE TABLE kyc AS SELECT * FROM kyc_final')
        c.unregister('kyc_final')
        k.to_parquet(p / 'node_mapping.parquet', index=False)
        dec = lambda x: f'coalesce(try_cast({x} AS DECIMAL(18,8))::DECIMAL(38,8),try_cast({x} AS DECIMAL(38,8)))'
        split = f"CASE WHEN ts<{sql_string(cfg.train_start)}::TIMESTAMP THEN 'calibration' WHEN ts<{sql_string(cfg.val_start)}::TIMESTAMP THEN 'train' WHEN ts<{sql_string(cfg.test_start)}::TIMESTAMP THEN 'val' WHEN ts<{sql_string(cfg.test_end)}::TIMESTAMP THEN 'test' ELSE 'outside' END"
        c.execute(f"CREATE OR REPLACE TABLE tx AS SELECT *,cast(ts AS DATE) AS day,{split} AS split FROM (\n            SELECT r.edge_id,strptime(timestamp_raw,'%Y/%m/%d %H:%M') AS ts,s.node_id AS src,d.node_id AS dst,\n            {dec('amount_paid_raw')} AS paid,{dec('amount_received_raw')} AS received,\n            payment_currency,receiving_currency,payment_format,cast(label_raw AS TINYINT) AS y,duplicate_group_id\n            FROM retained r JOIN nodes s ON s.bank={bank('r.from_bank')} AND s.account=r.from_account\n            JOIN nodes d ON d.bank={bank('r.to_bank')} AND d.account=r.to_account)")
        total = c.execute('SELECT count(*) FROM retained').fetchone()[0]
        assert total == c.execute('SELECT count(*) FROM tx').fetchone()[0], 'Node join changed edge count'
        records = []
        attempt = -1
        ptype = None
        with files['patterns'].open(newline='') as f:
            for line_no, line in enumerate(f, 1):
                raw = line.rstrip('\r\n')
                s = raw.strip()
                if not s:
                    continue
                if s.startswith('BEGIN'):
                    if ptype is not None:
                        raise ValueError('Nested pattern block')
                    m = re.fullmatch('BEGIN LAUNDERING ATTEMPT - ([^:]+)(?::.*)?', s)
                    if not m:
                        raise ValueError(f'Bad BEGIN line {line_no}')
                    attempt += 1
                    ptype = m.group(1).strip()
                elif s.startswith('END'):
                    if ptype is None:
                        raise ValueError('Unmatched END')
                    ptype = None
                else:
                    vals = next(csv.reader([raw]))
                    if ptype is None or len(vals) != 11:
                        raise ValueError(f'Bad pattern transaction {line_no}')
                    records.append(dict(zip(RAW_COLUMNS, vals), attempt_id=attempt, pattern_type=ptype, pattern_row_id=len(records)))
        if ptype is not None:
            raise ValueError('Unclosed pattern block')
        pat = pd.DataFrame(records, columns=RAW_COLUMNS + ['attempt_id', 'pattern_type', 'pattern_row_id'])
        c.register('pattern_input', pat)
        on = ' AND '.join(('r.' + x + '=p.' + x for x in RAW_COLUMNS))
        matched = c.execute(f'SELECT r.edge_id,p.attempt_id,p.pattern_type,p.pattern_row_id FROM pattern_input p\n            LEFT JOIN retained r ON {on}').fetchdf()
        if matched.edge_id.isna().any():
            raise ValueError('Unmatched pattern rows; inspect source formatting instead of silently changing labels')
        assert not matched.pattern_row_id.duplicated().any(), 'Ambiguous pattern transaction match'
        assert not matched.edge_id.duplicated().any(), 'One edge belongs to multiple pattern records'
        matched['edge_id'] = matched.edge_id.astype('int64')
        matched.to_parquet(p / 'pattern_labels.parquet', index=False)
        c.execute(f'COPY (SELECT * FROM tx ORDER BY ts,edge_id) TO {sql_string(p / 'edges.parquet')} (FORMAT PARQUET,COMPRESSION SNAPPY)')
        summary = c.execute('SELECT split,count(*) AS rows,sum(y) AS positives,min(ts) AS first_ts,max(ts) AS last_ts FROM tx GROUP BY split ORDER BY first_ts').fetchdf()
        summary.to_csv(p / 'dataset_summary.csv', index=False)
        for name, spec in specs.items():
            assert file_fingerprint(files[name]) == spec, 'Source changed during ingest'
        c.execute('DROP TABLE raw_hash')
        c.execute('DROP TABLE retained')
        c.execute('DROP TABLE raw')
        c.execute('CHECKPOINT')
        ingest_marker.unlink(missing_ok=True)
        stage_manifest(cfg, 'dataset', {'sources': specs, 'rows': total, 'nodes': len(k), 'kyc_unmatched_nodes': int(k.kyc_missing.sum()), 'duplicate_extra_rows': int(c.execute('SELECT coalesce(sum(multiplicity-1),0) FROM duplicate_groups').fetchone()[0]), 'node_map': file_fingerprint(p / 'node_mapping.parquet'), 'pattern_labels': file_fingerprint(p / 'pattern_labels.parquet')})
        return summary
    finally:
        c.close()

## `build_pairagg`

당일 시작 전까지의 계좌 쌍별 거래 이력을 계산합니다. 현재 날짜의 거래나 정답 레이블을 이력에 넣지 않습니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `{'pair_days': count}`

In [ ]:
def build_pairagg(cfg=Config()):
    """Per-pair day-start snapshot: excludes the whole current day, no label input."""
    require_stage(cfg, 'dataset')
    fresh_stage(cfg, 'pairagg')
    c = connect(cfg)
    p = base(cfg)
    try:
        c.execute('CREATE OR REPLACE TABLE pair_daily_state AS SELECT src,dst,day,\n          sum(n) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS history_count,\n          max(last_ts) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS last_ts\n          FROM (SELECT src,dst,day,count(*) AS n,max(ts) AS last_ts FROM tx GROUP BY src,dst,day)')
        c.execute("CREATE OR REPLACE TABLE pair_features AS SELECT t.src,t.dst,t.day,\n          coalesce(f.history_count,0)::BIGINT AS pair_history_count,\n          coalesce(r.history_count,0)::BIGINT AS reverse_history_count,\n          CASE WHEN f.last_ts IS NULL THEN -1 ELSE date_diff('minute',f.last_ts,t.day::TIMESTAMP) END AS pair_age_at_day_start\n          FROM (SELECT DISTINCT src,dst,day FROM tx) t\n          ASOF LEFT JOIN pair_daily_state f ON t.src=f.src AND t.dst=f.dst AND t.day>f.day\n          ASOF LEFT JOIN pair_daily_state r ON t.src=r.dst AND t.dst=r.src AND t.day>r.day")
        assert c.execute('SELECT count(*) FROM pair_features WHERE pair_history_count<0 OR reverse_history_count<0 OR pair_age_at_day_start< -1').fetchone()[0] == 0
        c.execute(f'COPY pair_features TO {sql_string(p / 'pair_daily_features.parquet')} (FORMAT PARQUET)')
        count = c.execute('SELECT count(*) FROM pair_features').fetchone()[0]
        c.execute('CHECKPOINT')
        stage_manifest(cfg, 'pairagg', {'pair_days': count, 'as_of': 'strictly before day start', 'label_inputs': False})
        return {'pair_days': count}
    finally:
        c.close()

## `build_features`

학습 이전 보정 구간으로 통화별 기준을 만들고 거래 간격·빈도·계좌 관계 피처를 계산합니다.

**입력:** `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `{'rows': n, 'edge_features': EDGE_FEATURES, 'node_features': NODE_FEATURES}`

In [ ]:
def build_features(cfg=Config()):
    """Strictly prior-minute rhythm + prior-day pair state + label-free KYC.
    Currency baselines are fitted in a dedicated period BEFORE training starts.
    No daily input feature uses transaction labels or future days.
    """
    require_stage(cfg, 'dataset')
    require_stage(cfg, 'pairagg')
    fresh_stage(cfg, 'features')
    c = connect(cfg)
    p = base(cfg)
    try:
        c.execute(f'CREATE OR REPLACE TEMP TABLE calibration_amounts AS SELECT * FROM (\n           SELECT payment_currency AS currency,paid::DOUBLE AS a FROM tx WHERE ts>={sql_string(cfg.calibration_start)}::TIMESTAMP AND ts<{sql_string(cfg.train_start)}::TIMESTAMP\n           UNION ALL SELECT receiving_currency,received::DOUBLE FROM tx WHERE ts>={sql_string(cfg.calibration_start)}::TIMESTAMP AND ts<{sql_string(cfg.train_start)}::TIMESTAMP\n          )')
        c.execute('CREATE OR REPLACE TABLE currency_stats AS WITH scales AS (SELECT currency,median(a) FILTER(WHERE a>0) AS scale FROM calibration_amounts GROUP BY currency), z AS (SELECT c.currency,s.scale,ln(1+c.a/s.scale) AS a FROM calibration_amounts c JOIN scales s USING(currency)) SELECT currency,scale,median(a) AS med,greatest(quantile_cont(a,.75)-quantile_cont(a,.25),.001) AS iqr FROM z GROUP BY currency,scale')
        stats = c.execute('SELECT * FROM currency_stats ORDER BY currency').fetchdf()
        if stats.empty:
            raise ValueError('Calibration period has no transactions')
        finite('currency baseline', stats[['scale', 'med', 'iqr']].to_numpy(float))
        if (stats.scale <= 0).any():
            raise ValueError('No positive calibration scale for currency')
        global_med = float(stats.med.median())
        global_iqr = float(stats.iqr.median())
        stats.to_parquet(p / 'currency_stats.parquet', index=False)
        json_write(p / 'currency_fallback.json', {'med': global_med, 'iqr': global_iqr, 'method': 'unseen currency receives neutral amount z=0; separate missing-baseline flags are emitted'})
        defs = []
        for key, part in [('pair', 'src,dst'), ('sender', 'src'), ('receiver', 'dst')]:
            defs += [f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) AS {key}_prior_count", f"max(ts) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) AS {key}_last_ts", f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING) AS {key}_count_1h"]
        c.execute('CREATE OR REPLACE TABLE temporal AS SELECT edge_id,' + ','.join(defs) + ' FROM tx')
        gap = lambda name: f"CASE WHEN z.{name}_last_ts IS NULL THEN 0 ELSE ln(2+date_diff('minute',z.{name}_last_ts,t.ts)) END"
        pz = 'CASE WHEN ps.scale IS NULL THEN 0 ELSE greatest(-8,least(8,(ln(1+t.paid::DOUBLE/ps.scale)-ps.med)/ps.iqr)) END'
        rz = 'CASE WHEN rs.scale IS NULL THEN 0 ELSE greatest(-8,least(8,(ln(1+t.received::DOUBLE/rs.scale)-rs.med)/rs.iqr)) END'
        expr = ['ln(1+z.pair_prior_count)', gap('pair'), 'ln(1+z.pair_count_1h)', gap('sender'), 'ln(1+z.sender_count_1h)', gap('receiver'), 'ln(1+z.receiver_count_1h)', pz, 'ln(1+p.pair_history_count)', 'ln(1+p.reverse_history_count)', 'CASE WHEN p.pair_age_at_day_start<0 THEN 0 ELSE ln(2+p.pair_age_at_day_start) END', 'CASE WHEN sk.entity_id IS NOT NULL AND sk.entity_id=dk.entity_id THEN 1 ELSE 0 END', "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' AND sk.country=dk.country THEN 1 ELSE 0 END", "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' THEN 1 ELSE 0 END", 'CASE WHEN t.payment_currency=t.receiving_currency THEN 1 ELSE 0 END', 'CASE WHEN t.src=t.dst THEN 1 ELSE 0 END', 'CASE WHEN ps.scale IS NULL THEN 1 ELSE 0 END', 'CASE WHEN rs.scale IS NULL THEN 1 ELSE 0 END']
        select = ','.join((f'cast(({v}) AS FLOAT) AS {name}' for name, v in zip(EDGE_FEATURES, expr)))
        c.execute(f'CREATE OR REPLACE TABLE features AS SELECT t.edge_id,t.ts,t.day,t.src,t.dst,{select},\n            cast(({rz}) AS FLOAT) AS received_currency_z\n          FROM tx t JOIN temporal z USING(edge_id) JOIN pair_features p ON t.src=p.src AND t.dst=p.dst AND t.day=p.day\n          JOIN kyc sk ON sk.node_id=t.src JOIN kyc dk ON dk.node_id=t.dst\n          LEFT JOIN currency_stats ps ON ps.currency=t.payment_currency LEFT JOIN currency_stats rs ON rs.currency=t.receiving_currency')
        bad = ' OR '.join((f'{x} IS NULL OR NOT isfinite({x})' for x in EDGE_FEATURES + ['received_currency_z']))
        nbad = c.execute('SELECT count(*) FROM features WHERE ' + bad).fetchone()[0]
        if nbad:
            raise ValueError(f'{nbad} nonfinite feature rows; no automatic imputation')
        n = c.execute('SELECT count(*) FROM features').fetchone()[0]
        assert n == c.execute('SELECT count(*) FROM tx').fetchone()[0]
        assert c.execute('SELECT count(*) FROM (SELECT edge_id FROM features GROUP BY edge_id HAVING count(*)!=1)').fetchone()[0] == 0
        c.execute(f'COPY (SELECT * FROM features ORDER BY ts,edge_id) TO {sql_string(p / 'edge_features.parquet')} (FORMAT PARQUET,COMPRESSION SNAPPY)')
        c.execute('DROP TABLE temporal')
        c.execute('CHECKPOINT')
        stage_manifest(cfg, 'features', {'rows': n, 'edge_features': EDGE_FEATURES, 'node_features': NODE_FEATURES, 'currency_fit_before': cfg.train_start, 'label_inputs': False, 'edge_alignment': 'explicit edge_id join, not row position'})
        return {'rows': n, 'edge_features': EDGE_FEATURES, 'node_features': NODE_FEATURES}
    finally:
        c.close()

## `DayData`

하루 그래프의 계좌 ID·특징, 연결, 대상 거래, 정답 배열을 저장합니다.

**메서드:** . 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
@dataclass
class DayData:
    day: str
    node_ids: np.ndarray
    node_x: np.ndarray
    context_src: np.ndarray
    context_dst: np.ndarray
    context_x: np.ndarray
    target_ids: np.ndarray
    target_src: np.ndarray
    target_dst: np.ndarray
    target_x: np.ndarray
    labels: np.ndarray | None

## `load_day`

하루의 판정 대상 거래와 과거~당일까지의 문맥 거래를 모아 그래프를 만듭니다. with_labels는 정답을 별도 배열로 읽을지 지정합니다.

**입력:** `cfg`, `day`, `with_labels`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `DayData(str(day.date()), node_ids, nx, context.src.to_numpy(np.int64), context.dst.to_numpy(np.int64), finite('context edges', context[EDGE_FEATURES].to_numpy(np.float32)), targets`

In [ ]:
def load_day(cfg, day, with_labels=False):
    """Fixed daily-close context. Targets are this day's transactions only.
    Context edges are selected once per recipient, not once per target batch.
    Node statistics use all window transactions, in currency-normalized units.
    Labels are loaded only when explicitly requested by training/evaluation.
    """
    require_stage(cfg, 'features')
    day = pd.Timestamp(day).normalize()
    start = day - pd.Timedelta(days=cfg.history_days - 1)
    if day < pd.Timestamp(cfg.train_start):
        raise ValueError('No predictions during baseline calibration')
    c = connect(cfg, read_only=True)
    try:
        c.execute(f'CREATE TEMP VIEW hist AS SELECT f.*,t.payment_currency,t.receiving_currency,\n           ks.entity_id AS source_entity,kd.entity_id AS target_entity\n           FROM features f JOIN tx t USING(edge_id) JOIN kyc ks ON ks.node_id=f.src JOIN kyc kd ON kd.node_id=f.dst\n           WHERE f.day>={sql_string(str(start.date()))}::DATE AND f.day<={sql_string(str(day.date()))}::DATE')
        agg = c.execute('WITH outgoing AS (SELECT src AS node_id,count(*) AS out_n,count(DISTINCT dst) AS out_u,\n              count(*) FILTER(WHERE src=dst) AS self_n,avg(paid_currency_z) AS out_z,\n              avg(CASE WHEN abs(paid_currency_z)>=3 THEN 1.0 ELSE 0 END) AS out_tail FROM hist GROUP BY src),\n          incoming AS (SELECT dst AS node_id,count(*) AS in_n,count(DISTINCT src) AS in_u,avg(received_currency_z) AS in_z,\n              avg(CASE WHEN abs(received_currency_z)>=3 THEN 1.0 ELSE 0 END) AS in_tail FROM hist GROUP BY dst),\n          pairs AS (SELECT DISTINCT src,dst FROM hist WHERE src!=dst),\n          reciprocal AS (SELECT a.src AS node_id,count(*) AS reciprocal_n FROM pairs a JOIN pairs b ON a.src=b.dst AND a.dst=b.src GROUP BY a.src),\n          currency_set AS (SELECT src AS node_id,payment_currency AS currency FROM hist UNION SELECT dst,receiving_currency FROM hist),\n          currencies AS (SELECT node_id,count(*) AS n_currency FROM currency_set GROUP BY node_id),\n          active AS (SELECT src AS node_id,day FROM hist UNION SELECT dst,day FROM hist),\n          active_days AS (SELECT node_id,count(*) AS n_days FROM active GROUP BY node_id),\n          ent_pairs AS (SELECT DISTINCT source_entity,target_entity FROM hist WHERE source_entity IS NOT NULL AND target_entity IS NOT NULL),\n          ent_out AS (SELECT source_entity AS entity_id,count(*) AS ent_out_n FROM ent_pairs GROUP BY source_entity),\n          ent_in AS (SELECT target_entity AS entity_id,count(*) AS ent_in_n FROM ent_pairs GROUP BY target_entity)\n          SELECT k.*,CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER(PARTITION BY k.entity_id) END AS observed_entity_accounts,coalesce(o.out_n,0) AS out_n,coalesce(i.in_n,0) AS in_n,coalesce(o.out_u,0) AS out_u,\n           coalesce(i.in_u,0) AS in_u,coalesce(o.self_n,0) AS self_n,coalesce(r.reciprocal_n,0) AS reciprocal_n,\n           coalesce(o.out_z,0) AS out_z,coalesce(i.in_z,0) AS in_z,coalesce(o.out_tail,0) AS out_tail,\n           coalesce(i.in_tail,0) AS in_tail,cu.n_currency,ad.n_days,coalesce(eo.ent_out_n,0) AS ent_out_n,\n           coalesce(ei.ent_in_n,0) AS ent_in_n\n          FROM active_days ad JOIN kyc k USING(node_id) LEFT JOIN outgoing o USING(node_id)\n          LEFT JOIN incoming i USING(node_id) LEFT JOIN reciprocal r USING(node_id)\n          JOIN currencies cu USING(node_id) LEFT JOIN ent_out eo ON k.entity_id=eo.entity_id\n          LEFT JOIN ent_in ei ON k.entity_id=ei.entity_id ORDER BY node_id').fetchdf()
        if agg.empty:
            raise ValueError(f'No graph history on {day.date()}')
        v = lambda name: agg[name].to_numpy(np.float64)
        outn, inn = (v('out_n'), v('in_n'))
        ou, iu = (v('out_u'), v('in_u'))
        rec = v('reciprocal_n')
        fields = [np.log1p(outn), np.log1p(inn), np.log1p(ou), np.log1p(iu), np.log1p(v('self_n')), np.log1p(rec), (outn - inn) / np.maximum(outn + inn, 1), rec / np.maximum(ou + iu, 1), v('out_z'), v('in_z'), v('out_tail'), v('in_tail'), np.log1p(v('n_currency')), np.log1p(v('n_days'))]
        fields += [(agg.entity_type == typ).to_numpy(np.float64) for typ in ENTITY_TYPES]
        fields += [np.log1p(v('observed_entity_accounts')), np.log1p(v('ent_out_n')), np.log1p(v('ent_in_n'))]
        nx = finite('daily node features', np.stack(fields, axis=1).astype(np.float32))
        node_ids = agg.node_id.to_numpy(np.int64)
        names = ','.join(EDGE_FEATURES)
        context = c.execute(f'SELECT edge_id,src,dst,{names} FROM hist\n          QUALIFY row_number() OVER(PARTITION BY dst ORDER BY ts DESC,edge_id DESC)<={cfg.incoming_neighbors}\n          ORDER BY dst,ts DESC,edge_id DESC').fetchdf()
        targets = c.execute(f'SELECT f.edge_id,f.src,f.dst,{','.join(('f.' + x for x in EDGE_FEATURES))}\n          {(',t.y' if with_labels else '')} FROM features f {('JOIN tx t USING(edge_id)' if with_labels else '')}\n          WHERE f.day={sql_string(str(day.date()))}::DATE ORDER BY f.edge_id').fetchdf()
        if targets.empty:
            raise ValueError(f'No target transactions on {day.date()}')
        return DayData(str(day.date()), node_ids, nx, context.src.to_numpy(np.int64), context.dst.to_numpy(np.int64), finite('context edges', context[EDGE_FEATURES].to_numpy(np.float32)), targets.edge_id.to_numpy(np.int64), targets.src.to_numpy(np.int64), targets.dst.to_numpy(np.int64), finite('target edges', targets[EDGE_FEATURES].to_numpy(np.float32)), targets.y.to_numpy(np.int8) if with_labels else None)
    finally:
        c.close()

## `FixedNeighborSampler`

판정할 거래 양 끝 계좌에서 들어오는 이웃을 두 번 따라가 배치 그래프를 만듭니다. batch는 모델에 전달할 다섯 배열을 반환합니다.

**메서드:** `__init__`, `incoming`, `batch`. 클래스 내부 메서드는 같은 상태를 공유하므로 함께 정의합니다.

In [ ]:
class FixedNeighborSampler:
    """Deterministic two-hop incoming neighborhoods, independent of batch members.
    A fixed full incoming list is expanded for seeds then first-hop sources.
    This matches two-layer GNN outputs on the fixed sampled daily graph.
    """

    def __init__(self, day):
        self.day = day
        self.dest, self.start, self.count = np.unique(day.context_dst, return_index=True, return_counts=True)
        assert np.all(day.context_dst[:-1] <= day.context_dst[1:])

    def incoming(self, nodes):
        nodes = np.unique(nodes)
        i = np.searchsorted(self.dest, nodes)
        ok = i < len(self.dest)
        ok[ok] = self.dest[i[ok]] == nodes[ok]
        i = i[ok]
        if not len(i):
            return np.empty(0, np.int64)
        counts = self.count[i]
        starts = self.start[i]
        offsets = np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)
        return np.repeat(starts, counts) + offsets

    def batch(self, positions):
        d = self.day
        positions = np.asarray(positions, dtype=np.int64)
        seeds = np.unique(np.r_[d.target_src[positions], d.target_dst[positions]])
        first = self.incoming(seeds)
        second = self.incoming(d.context_src[first])
        ci = np.union1d(first, second)
        nodes = np.unique(np.r_[seeds, d.context_src[ci], d.context_dst[ci]])
        ni = np.searchsorted(d.node_ids, nodes)
        assert (ni < len(d.node_ids)).all() and np.array_equal(d.node_ids[ni], nodes), 'Unknown node mapping'
        ei = np.stack([np.searchsorted(nodes, d.context_src[ci]), np.searchsorted(nodes, d.context_dst[ci])])
        targets = np.stack([np.searchsorted(nodes, d.target_src[positions]), np.searchsorted(nodes, d.target_dst[positions])])
        return (d.node_x[ni], ei, d.context_x[ci], targets, d.target_x[positions])

## `torch_modules`

현재 커널에 설치된 PyTorch와 PyTorch Geometric을 가져옵니다.

**입력:** 없음.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `(torch, TransformerConv)`

In [ ]:
def torch_modules():
    import torch
    from torch_geometric.nn import TransformerConv
    return (torch, TransformerConv)

## `tensor_batch`

배치 배열을 PyTorch 텐서로 변환합니다. 인덱스는 정수, 피처는 실수로 지정합니다.

**입력:** `batch`, `device`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `tuple((torch.as_tensor(x, dtype=torch.long if i in {1, 3} else torch.float32, device=device) for i, x in enumerate(batch)))`

In [ ]:
def tensor_batch(batch, device):
    torch, _ = torch_modules()
    return tuple((torch.as_tensor(x, dtype=torch.long if i in {1, 3} else torch.float32, device=device) for i, x in enumerate(batch)))

## `split_days`

설정한 반열린 날짜 구간에서 날짜 목록을 생성합니다.

**입력:** `cfg`, `split`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `[str(x[0]) for x in c.execute('SELECT DISTINCT day FROM tx WHERE split=? ORDER BY day', [split]).fetchall()]`

In [ ]:
def split_days(cfg, split):
    c = connect(cfg, read_only=True)
    try:
        return [str(x[0]) for x in c.execute('SELECT DISTINCT day FROM tx WHERE split=? ORDER BY day', [split]).fetchall()]
    finally:
        c.close()

## `select_threshold`

검증 정밀도와 재현율로 F1을 계산해 최대 F1의 임계값을 선택합니다.

**입력:** `validation`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `float(threshold[int(np.argmax(f))])`

In [ ]:
def select_threshold(validation):
    from sklearn.metrics import precision_recall_curve
    if validation.y.nunique() != 2:
        raise ValueError('Validation must contain both labels to select an F1 threshold')
    p, r, threshold = precision_recall_curve(validation.y, validation.score)
    f = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(threshold[int(np.argmax(f))])

## `evaluate_scores`

정답과 예측 점수로 AP·정밀도·재현율·F1·혼동행렬을 계산합니다.

**입력:** `frame`, `threshold`, `pattern_labels`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `metrics`

In [ ]:
def evaluate_scores(frame, threshold, pattern_labels=None):
    from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
    finite('evaluation scores', frame.score.to_numpy())
    if not frame.y.isin([0, 1]).all():
        raise ValueError('Invalid evaluation label')
    y = frame.y.to_numpy(np.int8)
    pred = (frame.score.to_numpy() >= threshold).astype(np.int8)
    positives = int(y.sum())
    rank = np.argsort(-frame.score.to_numpy(), kind='stable')
    metrics = {'threshold': threshold, 'rows': len(frame), 'positives': positives, 'AP': float(average_precision_score(y, frame.score)) if positives else None, 'precision': float(precision_score(y, pred, zero_division=0)), 'recall': float(recall_score(y, pred, zero_division=0)), 'F1': float(f1_score(y, pred, zero_division=0)), 'confusion_matrix': confusion_matrix(y, pred, labels=[0, 1]).tolist(), 'recall_at_1_percent': float(y[rank[:max(1, int(np.ceil(len(y) * 0.01)))]].sum() / positives) if positives else None, 'alarms': int(pred.sum()), 'threshold_source': 'validation only', 'pattern_recall': {}}
    if pattern_labels is not None:
        pl = pd.read_parquet(pattern_labels)
        z = frame.assign(pred=pred).merge(pl[['edge_id', 'pattern_type']], on='edge_id', how='left', validate='one_to_one')
        z = z[z.y == 1].copy()
        z['pattern_type'] = z.pattern_type.fillna('패턴 파일 미연결 양성')
        metrics['pattern_recall'] = {k: {'positives': len(g), 'recall': float(g.pred.mean())} for k, g in z.groupby('pattern_type')}
    return metrics

## 다음 단계에 연결

실행한 코드 셀을 내부 모듈로 저장합니다. 내용을 수정했다면 이 셀을 다시 실행하고 통합 노트북의 커널을 재시작하세요.

In [ ]:
export_cells('10_데이터전처리_함수설명.ipynb', 'data')